In [1]:
import numpy as np
import string

In [3]:
np.random.seed(1234)
# setting seed to generate the same random number

initial → frequency distribution of first words in lines.

first_order → maps one word to possible next words (bigram).

second_order → maps two words to possible next words (trigram).

In [4]:
initial = {} # start of a phase
first_order = {} # second word only
second_order = {}

In [7]:
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [5]:
def remove_punctuation(s):
  return s.translate(str.maketrans('','',string.punctuation))

str.maketrans(x, y, z) creates a translation table.

The third argument (z) is a list of characters to delete.

So here: str.maketrans('', '', string.punctuation) creates a mapping that says: delete every punctuation character.

In [9]:
!wget -nc https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/robert_frost.txt

--2025-09-29 19:13:24--  https://raw.githubusercontent.com/lazyprogrammer/machine_learning_examples/master/hmm_class/robert_frost.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 56286 (55K) [text/plain]
Saving to: ‘robert_frost.txt’

robert_frost.txt    100%[===================>]  54.97K  --.-KB/s    in 0.02s   

2025-09-29 19:13:25 (3.06 MB/s) - ‘robert_frost.txt’ saved [56286/56286]



the key to the dictionary can be a tuple like: I am

I am happy

I am sad

I am happy

key-> I am

In [10]:
def add2dict(d,k,v):
  if k not in d:
    d[k]= []
  d[k].append(v)

d → a dictionary you’re adding data into.

k → the key you want to add to the dictionary.

v → the value you want to associate with that key.

Steps:

if k not in d: → check if the key already exists in the dictionary.

d[k] = [] → if the key does not exist, create a new empty list at that key.

d[k].append(v) → add (append) the value into the list for that key.

In [26]:
# Frost training code

In [11]:
# https://chatgpt.com/c/68daddad-5dec-8322-9637-701860adfd34
for line in open('robert_frost.txt'):
  tokens = remove_punctuation(line.rstrip().lower()).split()

  T = len(tokens)
  for i in range(T):
    t = tokens[i]
    if i == 0:
      # measure the distribution of the first word
      initial[t] = initial.get(t, 0.) + 1
      # initial is a dictionary: {word → count}.
      # dict.get(key, default_value)
    else:
      t_1 = tokens[i-1]
      if i == T - 1:
        # measure probability of ending the line
        add2dict(second_order, (t_1, t), 'END')
      if i == 1:
        # measure distribution of second word
        # given only first word
        add2dict(first_order, t_1, t)
        # third word or later
      else:
        t_2 = tokens[i-2]
        add2dict(second_order, (t_2, t_1), t)

In [13]:
# normalize the distributions
initial_total = sum(initial.values())
# t is key and c is value
# initial.items() will show the tuple of dictionary initial
for t, c in initial.items():
    initial[t] = c / initial_total

In [14]:
# convert [cat, cat, cat, dog, dog, dog, dog, mouse, ...]
# into {cat: 0.5, dog: 0.4, mouse: 0.1}

def list2pdict(ts):
  # turn each list of possibilities into a dictionary of probabilities
  d = {}
  n = len(ts)
  for t in ts:
    d[t] = d.get(t, 0.) + 1
  for t, c in d.items():
    d[t] = c / n
  return d

In [15]:
for t_1, ts in first_order.items():
  # replace list with dictionary of probabilities
  first_order[t_1] = list2pdict(ts)

In [17]:
for k, ts in second_order.items():
  second_order[k] = list2pdict(ts)

In [18]:
def sample_word(d):
  # print "d:", d
  p0 = np.random.random()
  # print "p0:", p0
  cumulative = 0
  for t, p in d.items():
    cumulative += p
    if p0 < cumulative:
      return t
  assert(False) # should never get here

In [19]:
def generate():
  for i in range(4): # generate 4 lines
    sentence = []

    # initial word
    w0 = sample_word(initial)
    sentence.append(w0)

    # sample second word
    w1 = sample_word(first_order[w0])
    sentence.append(w1)

    # second-order transitions until END
    while True:
      w2 = sample_word(second_order[(w0, w1)])
      if w2 == 'END':
        break
      sentence.append(w2)
      w0 = w1
      w1 = w2
    print(' '.join(sentence))

In [23]:
generate()

up to pass a winter eve
to make them out
and then someone
dyou know a person so related to herself


In [21]:
# Exercise:
#
# Determine the vocabulary size (V)
# We know that pi has shape V, A1 has shape V x V, and A2 has shape V x V x V
#
# In comparison, how many values are stored in our dictionaries?

In [24]:
def compare_storage(initial, first_order, second_order):
    # Vocabulary size
    V = len(set(list(initial.keys()) +
                list(first_order.keys()) +
                [w for (w1, w2) in second_order.keys() for w in (w1, w2)]))

    print("Vocabulary size (V):", V)
    print("Matrix model values needed:")
    print("  pi:", V)
    print("  A1:", V * V)
    print("  A2:", V * V * V)
    print("  Total:", V + V*V + V*V*V)

    print("\nDictionary model values stored:")
    print("  initial:", len(initial))
    print("  first_order:", sum(len(v) for v in first_order.values()))
    print("  second_order:", sum(len(v) for v in second_order.values()))
    print("  Total:", len(initial) +
                    sum(len(v) for v in first_order.values()) +
                    sum(len(v) for v in second_order.values()))


In [25]:
# Exercise 2:
# We can skip the step where we accumulate all the possible next words in a list
# E.g. [cat, cat, dog, dog, dog, ...]
#
# Instead, like we do with the initial state distribution, create the dictionary
# of counts directly as you loop through the data.
#
# You'll no longer need list2pdict()

In [29]:
import re # python regular expression
import numpy as np

def remove_punctuation(s):
    return re.sub(r"[^\w\s]", "", s)
    # re.sub(pattern, repl, string) → replace matches

def add_count_dict(d, k, v):
    """Accumulate counts directly: d[k][v] += 1."""
    if k not in d:
        d[k] = {}
    d[k][v] = d[k].get(v, 0) + 1

# === Build models (counts) ===
initial = {}        # first-word counts: {w0: count}
first_order = {}    # bigram counts: {w0: {w1: count}}
second_order = {}   # trigram counts: {(w0,w1): {w2: count}}

with open("robert_frost.txt", encoding="utf-8") as f:
    for line in f:
        tokens = remove_punctuation(line.rstrip().lower()).split()
        T = len(tokens)
        if T == 0:
            continue

        # initial word
        w0 = tokens[0]
        initial[w0] = initial.get(w0, 0) + 1

        # bigram (second word given first)
        if T >= 2:
            add_count_dict(first_order, tokens[0], tokens[1])

        # trigram (w2 given (w0,w1)) for positions i >= 2
        for i in range(2, T):
            add_count_dict(second_order, (tokens[i-2], tokens[i-1]), tokens[i])

        # mark END after the last pair
        if T >= 2:
            add_count_dict(second_order, (tokens[-2], tokens[-1]), "END")

# === Normalize to probabilities ===
# initial -> probabilities
total_initial = sum(initial.values())
for w in list(initial.keys()):
    initial[w] = initial[w] / total_initial

# first_order -> probabilities
for w0, next_counts in first_order.items():
    s = sum(next_counts.values())
    for w1 in list(next_counts.keys()):
        next_counts[w1] = next_counts[w1] / s

# second_order -> probabilities
for pair, next_counts in second_order.items():
    s = sum(next_counts.values())
    for w2 in list(next_counts.keys()):
        next_counts[w2] = next_counts[w2] / s

def sample_word(pd):
    """Roulette-wheel sampling from {token: prob}."""
    p0 = np.random.random()
    cumulative = 0.0
    for t, p in pd.items():
        cumulative += p
        if p0 < cumulative:
            return t
    # Fallback in case of tiny FP error
    return next(iter(pd))

def generate():
    """Generate 4 lines using initial, bigram, trigram models."""
    for _ in range(4):
        sentence = []

        # sample first and second words
        w0 = sample_word(initial)
        sentence.append(w0)

        # if no bigram for w0, restart the line
        if w0 not in first_order:
            print(" ".join(sentence))
            continue

        w1 = sample_word(first_order[w0])
        sentence.append(w1)

        # keep sampling with trigram until END (or missing context)
        while True:
            ctx = (w0, w1)
            if ctx not in second_order:
                break
            w2 = sample_word(second_order[ctx])
            if w2 == "END":
                break
            sentence.append(w2)
            w0, w1 = w1, w2

        print(" ".join(sentence))




In [30]:
generate()

and this bill
my french indian esquimaux
the origin of all this now too much
though what a gentle lot we are
